# Welcome to RAG week!!

## Expert Knowledge Worker

### A question answering Assistant that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The AI assistant needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

This first implementation will use a simplistic, brute-force type of RAG..

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications of this week's projects</h2>
            <span style="color:#181;">RAG is perhaps the most immediately applicable technique of anything that we cover in the course! In fact, there are commercial products that do precisely what we build this week: nuanced querying across large databases of information, such as company contracts or product specs. RAG gives you a quick-to-market, low cost mechanism for adapting an LLM to your business area.</span>
        </td>
    </tr>
</table>

In [ ]:
# import libraries

import os
import glob
from dotenv import load_dotenv
import gradio as gr 
from pathlib import Path
from openai import OpenAI, AzureOpenAI
from App.config import azure_endpoint, api_version, headers


In [ ]:
# loading keys

load_dotenv(override=True)

openai_api_key = os.getenv('cd_api_key')
gemini_api_key = os.getenv('GOOGLE_API_KEY')
ollama_api_key = os.getenv('OLLAMA_API_KEY')

if openai_api_key:
    print('OpenAI API key found and looks good so far')
else:
    print('OpenAI API key issue')

if gemini_api_key:
    print('Gemini API key found and looks good so far')
else:
    print('Gemini API key issue')

if ollama_api_key:
    print('Ollama API key found and looks good so far')
else:
    print('Ollama API key issue')


In [ ]:
# Initialize the objects
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "https://ollama.com/v1"

openai = AzureOpenAI(api_key=openai_api_key, azure_endpoint=azure_endpoint, api_version=api_version)
gemini = OpenAI(base_url=gemini_url, api_key=gemini_api_key)
ollama = OpenAI(base_url=ollama_url, api_key=ollama_api_key)

gpt_4_nano = "gpt-4.1-nano"
gemini_3 = "gemini-3.1-flash-lite"
gpt_oss_120b = "gpt-oss:120b-cloud"

### Let's read in all employee data into a dictionary

In [ ]:
knowledge = {}

filenames = glob.glob('week5/knowledge-base/employees/*')

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, 'r', encoding='utf-8') as fs:
        knowledge[name.lower()] = fs.read()


In [ ]:
knowledge

In [ ]:
knowledge["lancaster"]

In [ ]:
filenames = glob.glob('week5/knowledge-base/products/*')

for filename in filenames:
    name = Path(filename).stem
    with open(filename, 'r', encoding='utf-8') as fs:
        knowledge[name.lower()] = fs.read()


In [ ]:
knowledge.keys()

In [ ]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [ ]:
def get_relevant_context_simple(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    relevant_context = []
    for word in words:
        if (word in knowledge):
            relevant_context.append(knowledge[word])
    return relevant_context


## But a more pythonic way:

In [ ]:
def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]


In [ ]:
get_relevant_context("Who is Lancaster?")

In [ ]:
get_relevant_context("Who is Lancaster and what is carllm?")

In [ ]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = 'There is no additional context relevant to the user"s question'
    else:
        result = 'The following additional context might be relevant to the user"s questions \n\n'
        result += "\n\n".join(relevant_context)
    return result

In [ ]:
print(additional_context("Who is Alex Lancaster?"))

### Models and clients

In [ ]:
models = ['gpt-4.1-nano', "gemini-3.1-flash-lite", "gpt-oss:120b-cloud"]

clients = {'gpt-4.1-nano': openai, "gpt-oss:120b-cloud": openai, "gemini-3.1-flash-lite": gemini}

In [ ]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{'role': 'system', 'content': system_message}] + history + [{'role': 'user', 'content': message}]

    response = openai.chat.completions.create(model=gpt_4_nano, messages=messages, extra_headers=headers)
    return response.choices[0].message.content



## Now we will bring this up in Gradio using the Chat interface -

A quick and easy way to prototype a chat with an LLM

In [ ]:
app = gr.ChatInterface(fn=chat, type='messages')
app.launch()